In [ ]:
import os
import platform
import warnings

def configure_cuda_allocator():
    # Bezpečné pre všetky OS:
    parts = ["max_split_size_mb:128"]

    # 'expandable_segments' zapni len na Linuxe (Windows to nepodporuje)
    if platform.system() == "Linux":
        parts.insert(0, "expandable_segments:True")

    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(parts)

warnings.filterwarnings("ignore", message=r".*expandable_segments not supported.*")
import re
import json
import math
import unicodedata
from datetime import datetime
from typing import TypedDict, List, Dict, Any, Tuple

import torch

import pandas as pd

from docx import Document
from striprtf.striprtf import rtf_to_text

from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langgraph.graph import StateGraph, END

# =========================
# KONFIGURÁCIA
# =========================
MODE = 'title'  # tento skript je primárne pre titulky

INPUT_DIR   = './Input'
OUTPUT_DIR  = './Output'
PERSIST_DIR = './chroma_store'

THESAURUS_XLSX = './Thesaurus/SK_Local_Thesaurus.xlsx'
THESAURUS_COL  = 'slovak_terms'

# Embeddings (multilingual)
EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

# LLM: primárny a fallback (pevné načítanie na štarte)
#LLM_PRIMARY  = 'nvidia/Llama-3.1-Nemotron-Nano-8B-v1'
LLM_PRIMARY = 'marcuscedricridia/8B-Nemotaur-IT'
LLM_FALLBACK  = 'unsloth/Llama-3.1-Nemotron-Nano-8B-v1'

# Generačné nastavenia (KV cache disciplinované)
MAX_NEW_TOKENS = 256
DO_SAMPLE      = True
TEMPERATURE    = 0.6
TOP_P          = 0.95
REPETITION_PENALTY = 1.3

# Pamäťové nastavenia pre PRIMÁRNY model (znižujú fragmentáciu VRAM)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")  # stabilnejšie VRAM chovanie
configure_cuda_allocator()
# Agentné brány
CONF_GATE   = 0.75
MAX_RETRIES = 3
RECURSION_LIMIT = 30

# Chunkovanie do vektorového úložiska
CHUNK_CHARS    = 1200
CHUNK_OVERLAP  = 200
MAX_CTX_SNIPPETS = 6       # koľko úryvkov ide do RAG kontextu
MAX_CTX_CHARS    = 2000    # limit pre context injection

# =========================
# UTIL: diakritika, stopwords, tezaurus
# =========================
def load_stopwords(path='./Input/stopword_dictionary.txt'):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            words = {line.strip().lower() for line in f if line.strip()}
        print(f'Loaded {len(words)} stopwords from {path}')
        return words
    except FileNotFoundError:
        print(f'No stopword file at {path}. Continuing without.')
        return set()
    except Exception as e:
        print(f'Failed to load stopwords: {e}')
        return set()

STOPWORDS = load_stopwords()

def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)

    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))

    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]

    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL):
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f'Thesaurus file not found: {xlsx_path}')
    df = pd.read_excel(xlsx_path, engine='openpyxl')
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')

    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', cell))

    seen, terms = set(), []
    for t in parts:
        t = t.strip()
        if not t: continue
        low = t.lower()
        if len(low) == 1 or low in STOPWORDS: continue
        if t not in seen:
            seen.add(t); terms.append(t)
    print(f'Thesaurus terms loaded: {len(terms)}')
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

# =========================
# EMBEDDINGS + CHROMA (CPU)
# =========================
emb_lc = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": "cpu"})

thesaurus_store = Chroma(
    collection_name='thesaurus',
    persist_directory=PERSIST_DIR,
    embedding_function=emb_lc,
)
corpus_store = Chroma(
    collection_name='session_corpus',
    persist_directory=PERSIST_DIR,
    embedding_function=emb_lc,
)

# Naplň tezaurus do vektorového úložiska (len chýbajúce ID)
existing_ids = set(thesaurus_store.get().get('ids', []))
to_texts, to_ids = [], []
for t in THESAURUS_TERMS:
    tid = f"term::{t}"
    if tid not in existing_ids:
        to_texts.append(t); to_ids.append(tid)
if to_texts:
    thesaurus_store.add_texts(texts=to_texts, ids=to_ids, metadatas=[{'type':'term'}]*len(to_texts))

# Clear session corpus
cur_ids = corpus_store.get().get('ids', [])
if cur_ids:
    corpus_store.delete(ids=cur_ids)

thesaurus_ret = thesaurus_store.as_retriever(search_kwargs={'k': 12})
corpus_ret    = corpus_store.as_retriever(search_kwargs={'k': 8})

# =========================
# INDEXÁCIA DOKUMENTOV (RAG)
# =========================
def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None:
                    buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None:
                    buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []

def chunk_text_chars(text: str, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    s = (text or "").strip()
    if not s: return []
    chunks, i, n = [], 0, len(s)
    while i < n:
        j = min(n, i + size)
        chunks.append(s[i:j].strip())
        if j == n: break
        i = max(i + size - overlap, j)  # posun
    return [c for c in chunks if c]

def index_session_corpus(input_dir=INPUT_DIR):
    files = sorted(os.listdir(input_dir), key=str.lower)
    total = 0
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith('.docx'):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith('.rtf'):
            secs = split_rtf_by_headers(p)
        else:
            continue
        for header, text in secs:
            if not text.strip(): continue
            chunks = chunk_text_chars(text)
            if not chunks: continue
            ids   = [f"{f}::{header}::{i}" for i in range(len(chunks))]
            metas = [{'file': f, 'header': header, 'idx': i, 'kind': 'chunk'} for i in range(len(chunks))]
            corpus_store.add_texts(texts=chunks, ids=ids, metadatas=metas)
            total += len(chunks)
    print(f'Indexed {total} chunks to session_corpus.')

index_session_corpus(INPUT_DIR)

# =========================
# LLM: pevné načítanie s fallbackom
# =========================
def load_primary_llm():
    qconf = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    model = AutoModelForCausalLM.from_pretrained(
        LLM_PRIMARY,
        quantization_config=qconf,
        device_map=device_map,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    tok = AutoTokenizer.from_pretrained(LLM_PRIMARY)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    gen = pipeline(
                    "text-generation",
                    model=model,
                    tokenizer=tok,
                    device_map=device_map,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=DO_SAMPLE,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    repetition_penalty = REPETITION_PENALTY,
                    return_full_text=False,
                    )
    return HuggingFacePipeline(pipeline=gen), f"primary:{LLM_PRIMARY}"

def load_fallback_llm():
    # Zruš env hacky pre fallback => čistý load
    for k in ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_FLASH_ATTENTION"):
        if k in os.environ:
            os.environ.pop(k)
    torch.cuda.empty_cache()

    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        LLM_FALLBACK,
        device_map=device_map,
        torch_dtype=dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    tok = AutoTokenizer.from_pretrained(LLM_FALLBACK)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    gen = pipeline(
                    "text-generation",
                    model=model,
                    tokenizer=tok,
                    device_map=device_map,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=DO_SAMPLE,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    repetition_penalty = REPETITION_PENALTY,
                    return_full_text=False,
                    )
    
    return HuggingFacePipeline(pipeline=gen), f"fallback:{LLM_FALLBACK}"

try:
    LLM, USED_MODEL = load_primary_llm()
    print(f"[LLM] Loaded {USED_MODEL}")
except Exception as e:
    print(f"[WARN] Primary model failed: {e}")
    LLM, USED_MODEL = load_fallback_llm()
    print(f"[LLM] Loaded {USED_MODEL}")

# =========================
# CPU SentenceTransformer na scoring (sim + coverage)
# =========================
EMBED_ST = SentenceTransformer(EMBED_MODEL, device="cpu")

def format_title(s: str) -> str:
    s = (s or '').strip()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[\.，。…]+$', '', s)
    return s

def terms_matched_in_text(text: str) -> set:
    hits = set()
    if not text: return hits
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        if max(a, b) > 0:
            hits.add(canon)
    return hits

def _title_len_score(title: str) -> float:
    n = len(title.split())
    if 3 <= n <= 12: return 1.0
    return max(0.0, 1.0 - abs(n - 7) * 0.1)

def _contains_canon(title: str, canon: str) -> bool:
    return strip_accents(canon).lower() in strip_accents(title).lower()

def score_title_candidate(title: str, section_text: str):
    title = format_title(title)
    if not title or not section_text:
        return 0.0, 0.0, 0.0
    tv = EMBED_ST.encode([title], convert_to_numpy=True, normalize_embeddings=True)[0]
    sv = EMBED_ST.encode([section_text], convert_to_numpy=True, normalize_embeddings=True)[0]
    sim = float((tv @ sv).item())
    sim_score = max(0.0, (sim + 1.0) / 2.0)
    in_text = terms_matched_in_text(section_text)
    covered = sum(1 for c in in_text if _contains_canon(title, c))
    cov_score = (covered / max(1, len(in_text))) if in_text else 0.0
    len_score = _title_len_score(title)
    final = 0.6 * sim_score + 0.3 * cov_score + 0.1 * len_score
    return final, cov_score, sim

# =========================
# RAG pomocné: retrieval + kontext
# =========================
def retrieve_hits(q: str, k: int = 6):
    docs = corpus_ret.invoke(q) or []
    docs = docs[:k]
    cites = []
    for d in docs[:2]:
        md = d.metadata or {}
        cites.append({
            "file": md.get("file", "?"),
            "header": md.get("header", "?"),
            "snippet": (d.page_content or "")[:180]
        })
    return docs, cites

def assemble_rag_context(queries: List[str], max_snippets=MAX_CTX_SNIPPETS, max_chars=MAX_CTX_CHARS) -> str:
    seen, snippets = set(), []
    for q in queries:
        docs, _ = retrieve_hits(q, k=4)
        for d in docs:
            s = (d.page_content or "").strip()
            if not s: continue
            key = hash(s[:200])
            if key in seen: continue
            seen.add(key)
            snippets.append(s)
            if len(snippets) >= max_snippets:
                break
        if len(snippets) >= max_snippets:
            break
    ctx = "\n\n---\n\n".join(snippets)
    if len(ctx) > max_chars:
        ctx = ctx[: max_chars//2] + "\n...\n" + ctx[- max_chars//2 :]
    return ctx

def occ_candidates(text: str, topk=6):
    occ = {}
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0:
            occ[canon] = cnt
    return [k for k,_ in sorted(occ.items(), key=lambda kv: (-kv[1], -len(kv[0])))[:topk]]

# =========================
# AGENTNÉ NÓDY (LangGraph)
# =========================
class AgentState(TypedDict):
    file: str
    header: str
    text: str
    loop: int
    queries: List[str]
    citations: List[Dict[str, Any]]
    candidates: List[str]
    scores: Dict[str, Dict[str, float]]
    final: Dict[str, Any]

def _signature(lst: list[str]) -> str:
    # stabilný podpis bez ohľadu na poradie duplicít
    return "|".join(x.strip().lower() for x in lst if x.strip())

def node_seed(state: AgentState):
    txt = state["text"]
    seeds = occ_candidates(txt, topk=6)
    hdr = state.get("header") or ""
    if hdr and hdr != "__DOCUMENT__":
        hdr_terms = [w.strip() for w in re.split(r'[,\-–—:;]', hdr) if len(w.strip()) >= 3]
        for w in hdr_terms:
            lw = w.lower()
            if lw not in seeds:
                seeds.append(lw)
    seeds = seeds[:8]
    return {
        "queries": seeds,
        "queries_sig": _signature(seeds),
        "candidates": [], "citations": [], "scores": {}, "final": {}, "loop": 0,
        "no_progress": False
    }

def node_retrieve(state: AgentState):
    citations = list(state.get("citations", []))
    for q in state["queries"]:
        _, cites = retrieve_hits(q, k=4)
        citations.extend(cites)
    return {"citations": citations}

def title_prompt(rag_ctx: str, th_terms: List[str], section_text: str) -> str:
    seed = "\n".join(th_terms[:30])
    return (
        "ÚLOHA: Navrhni 4–6 presných a zrozumiteľných právnych nadpisov (3–12 slov) k zadanému textu.\n"
        "Požiadavky: vecné, bez bodky na konci, žiadne úvodzovky. "
        "Preferuj termíny z tezauru, ale doplň kontextové slová podľa potreby.\n"
        "Vráť IBA JSON zoznam reťazcov (bez komentárov).\n\n"
        f"TEZAURUS (výber):\n{seed}\n\n"
        f"RAG KONTEXT (úryvky):\n{rag_ctx}\n\n"
        f"PÔVODNÝ TEXT (pre referenciu pri výbere slov):\n{section_text[:1000]}\n\n"
        "ODPOVEĎ:"
    )

def node_propose(state: AgentState):
    txt = state["text"]
    rag_ctx = assemble_rag_context(state["queries"])
    prompt = title_prompt(rag_ctx, CANON_TERMS, txt)
    out = LLM.invoke(prompt)
    try:
        js = json.loads(out.strip().split('[')[-1].split(']')[0].join(['[', ']']))
        cands = [format_title(s) for s in js if isinstance(s, str)]
    except Exception:
        rough = [w.strip(' "„”') for w in re.split(r'[\n;]+', out)]
        cands = [format_title(w) for w in rough if 3 <= len(w.split()) <= 12]

    # fallback, ak by LLM nevydal nič použiteľné
    if not cands:
        tops = occ_candidates(txt, topk=3)
        if tops:
            # jednoduché šablóny, aby mali 3–12 slov
            cands = [format_title(f"Ustanovenia o {tops[0]}"), 
                        format_title(f"Právna úprava a podmienky {tops[0]}")]
        else:
            cands = ["Právna kvalifikácia", "Základné podmienky a postup"]

    # dedupe
    seen, dedup = set(), []
    for t in cands:
        k = strip_accents(t).lower()
        if t and k not in seen:
            seen.add(k); dedup.append(t)
    return {"candidates": dedup[:6]}

def node_score(state: AgentState):
    txt = state["text"]
    scored = {}
    for t in state["candidates"]:
        s, cov, sim = score_title_candidate(t, txt)
        scored[t] = {"score": s, "coverage": cov, "best_sim": sim}
    return {"scores": scored}

def reflect_prompt(scored: Dict[str, Dict[str, float]], section_text: str) -> str:
    table = "\n".join([f"- {t}: score={v['score']:.3f}, cov={v['coverage']:.3f}, sim={v['best_sim']:.3f}"
                        for t, v in scored.items()])
    return (
        "Vyber JEDEN najvhodnejší právny nadpis zo zoznamu kandidátov na základe skóre a textu.\n"
        "Kritériá: presnosť, úplnosť, zrozumiteľnosť, 3–12 slov, bez bodky. "
        "Vráť LEN vybraný nadpis (bez komentára).\n\n"
        f"Kandidáti a skóre:\n{table}\n\n"
        f"Text (referencia):\n{section_text[:800]}\n\n"
        "NADPIS:"
    )

def node_reflect(state: AgentState):
    txt = state["text"]
    scored = state["scores"]
    if not scored:
        # žiadni kandidáti/žiadne skóre – vynúťme ukončenie s defaultom
        return {"final": {"title": "Právna kvalifikácia", "confidence": 0.0,
                            "method": f"agentic-rag-safe ({USED_MODEL})"}}

    prompt = reflect_prompt(scored, txt)
    out = LLM.invoke(prompt).strip()
    title = format_title(re.sub(r'^[\-\•\:\s]+','', out))
    if title not in scored:
        title = max(scored.items(), key=lambda kv: kv[1].get("score", 0.0))[0]
        title = format_title(title)
    conf = float(scored.get(title, {}).get("score", 0.0))
    return {"final": {"title": title, "confidence": conf, "method": f"agentic-rag ({USED_MODEL})"}}

def _expand_queries(state: AgentState):
    # Rozšír dopyty o synonymá z tezauru + slová z vybraného nadpisu
    chosen = state["final"].get("title", "")
    expansions = []
    for w in re.split(r'[\s,;:–—-]+', chosen):
        w = w.strip().lower()
        if not w or len(w) < 3: continue
        if w in TERM_MATCHERS:
            expansions += TERM_MATCHERS[w]['synonyms']
        expansions.append(w)
    # doplň aj slová z headera
    hdr = state.get("header")
    if hdr and hdr != "__DOCUMENT__":
        expansions += [w.strip().lower() for w in re.split(r'[,\-–—:;]', hdr) if len(w.strip()) > 2]
    # dedupe a limit
    dd, seen = [], set()
    for e in expansions:
        if e and e not in seen:
            seen.add(e); dd.append(e)
    return dd[:8]

def node_retry(state: AgentState):
    prev_sig = state.get("queries_sig", "")
    new_q = _expand_queries(state)

    # ak nič nové alebo podpis ostáva rovnaký → označ ako bez pokroku
    new_sig = _signature(new_q) if new_q else prev_sig
    no_progress = (not new_q) or (new_sig == prev_sig)

    # ak bez pokroku, posuňme loop až na MAX_RETRIES, aby sa gate ukončil v ďalšom kole
    new_loop = state["loop"] + 1 if not no_progress else max(state["loop"] + 1, MAX_RETRIES)

    return {
            "queries": new_q if new_q else state["queries"],
            "queries_sig": new_sig,
            "loop": new_loop,
            "final": {},
            "no_progress": no_progress
            }

def gate(state: AgentState):
    conf = float(state.get("final", {}).get("confidence", 0.0) or 0.0)
    loop = int(state.get("loop", 0))
    no_progress = bool(state.get("no_progress", False))
    if conf >= CONF_GATE or loop >= MAX_RETRIES or no_progress:
        return "end"
    return "retry"

# =========================
# GRAF
# =========================
graph = StateGraph(AgentState)
graph.add_node("seed",     node_seed)
graph.add_node("retrieve", node_retrieve)
graph.add_node("propose",  node_propose)
graph.add_node("score",    node_score)
graph.add_node("reflect",  node_reflect)
graph.add_node("retry",    node_retry)

graph.set_entry_point("seed")
graph.add_edge("seed", "retrieve")
graph.add_edge("retrieve", "propose")
graph.add_edge("propose", "score")
graph.add_edge("score", "reflect")
graph.add_conditional_edges("reflect", gate, {"retry": "retry", "end": END})
graph.add_edge("retry", "retrieve")

APP = graph.compile()

# =========================
# DRIVER
# =========================
def agent_run(file: str, header: str, text: str):
    state: AgentState = {"file": file, "header": header, "text": text,"loop": 0, "queries": [], "candidates": [], "citations": [], "scores": {}, "final": {},"queries_sig": "", "no_progress": False}
    out = APP.invoke(state, config={"recursion_limit": RECURSION_LIMIT})
    return out["final"], out.get("queries", []), out.get("citations", [])

def process_all(input_dir=INPUT_DIR):
    rows = []
    files = sorted(os.listdir(input_dir), key=str.lower)

    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith('.docx'):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith('.rtf'):
            secs = split_rtf_by_headers(p)
        else:
            continue
        if not secs:
            print(f'No text in {f}')
            continue

        for header, text in secs:
            if not text.strip(): continue
            final, used_q, _ = agent_run(f, header, text)
            rows.append({
                "file": f,
                "header": header,
                "suggested_title": final.get("title", ""),
                "confidence": round(final.get("confidence", 0.0), 3),
                "method": final.get("method", ""),
                "used_queries": "; ".join(used_q),
                "llm_model": USED_MODEL
            })
        print(f'Processed {f} with {len(secs)} sections.')

    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    df = pd.DataFrame(rows, columns=[
        "file","header","suggested_title","confidence","method","used_queries","llm_model"
    ])
    xlsx_path = os.path.join(OUTPUT_DIR, f"agentic_rag_titles_{today}.xlsx")
    try:
        df.to_excel(xlsx_path, index=False, engine="openpyxl")
        print(f"Saved: {xlsx_path}")
    except Exception as e:
        print(f"[WARN] Excel write failed: {e}")
    return df

if __name__ == "__main__":
    df = process_all(INPUT_DIR)
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))